# POSSM SBP pretraining comparison

Colab workflow for compute-matched SBP experiments. The active configuration uses Brain2Text24 `competition_train` alone for the Stage-1 baseline; switching the named recipe can instead pool Brain2Text24 with unlabeled Brain2Text25 `train`/`val`. Stage 2 always fine-tunes on Brain2Text24 and evaluates on its held-out competition test split.


In [ ]:
# Mount Drive and resolve cache/output roots.

from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
DEFAULT_CACHE_ROOT_SMOOTHED = DRIVE_ROOT / 'utah_ssl' / 'data' / 'cache_v1_sbpclip12500_fp16_smoothed'
DEFAULT_CACHE_ROOT_RAW = DRIVE_ROOT / 'utah_ssl' / 'data' / 'cache_v1_sbpclip12500_fp16_raw'
STAGE1_B2T25_CACHE_ROOT_RAW = DEFAULT_CACHE_ROOT_RAW
STAGE1_B2T25_CACHE_ROOT_SMOOTHED = DEFAULT_CACHE_ROOT_SMOOTHED
OUTPUT_ROOT = DRIVE_ROOT / 'utah_ssl' / 'outputs' / 'ssl_experiments' / 'possm_masked_reconstruction'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('DRIVE_ROOT:', DRIVE_ROOT)
print('DEFAULT_CACHE_ROOT_SMOOTHED exists:', DEFAULT_CACHE_ROOT_SMOOTHED.exists())
print('DEFAULT_CACHE_ROOT_RAW exists:', DEFAULT_CACHE_ROOT_RAW.exists())
print('STAGE1_B2T25_CACHE_ROOT_SMOOTHED exists:', STAGE1_B2T25_CACHE_ROOT_SMOOTHED.exists())
print('OUTPUT_ROOT:', OUTPUT_ROOT)


In [ ]:
# Repo import bootstrap (Colab).

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/ethan-read/utah-ssl.git'
REPO_DIR = Path('/content/utah-ssl')
EXPERIMENTS_DIR = REPO_DIR / 'analysis' / 'active' / 'ssl_experiments'
POSSM_DIR = REPO_DIR / 'analysis' / 'reference' / 'possm'
POSSM_SSL_DIR = POSSM_DIR / 'possm_ssl'
BENCHMARK_DIR = REPO_DIR / 'analysis' / 'active' / 'transfer_benchmark' / 'ssl_autoresearch'

os.chdir('/content')
if REPO_DIR.exists():
    print('Using existing repo:', REPO_DIR)
    try:
        repo_status = subprocess.run(
            ['git', '-C', str(REPO_DIR), 'status', '--porcelain'],
            check=True,
            capture_output=True,
            text=True,
        )
        if repo_status.stdout.strip():
            print('Skipping auto-pull because the Colab checkout has local changes.')
        else:
            subprocess.run(
                ['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'],
                check=True,
            )
            print('Updated existing repo checkout from origin/main.')
    except Exception as exc:
        print('warning: could not auto-update existing repo checkout:', exc)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
for import_path in (REPO_DIR, EXPERIMENTS_DIR, POSSM_DIR, BENCHMARK_DIR):
    import_path_str = str(import_path)
    if import_path_str not in sys.path:
        sys.path.insert(0, import_path_str)

os.environ['SSL_AUTORESEARCH_CACHE_ROOT'] = str(DEFAULT_CACHE_ROOT_RAW)
os.environ['SSL_AUTORESEARCH_OUTPUT_ROOT'] = str(OUTPUT_ROOT)

if not POSSM_SSL_DIR.exists():
    raise FileNotFoundError('Missing POSSM helper package in cloned repo checkout.')

print('cwd:', Path.cwd())
print('possm_ssl dir exists:', POSSM_SSL_DIR.exists(), POSSM_SSL_DIR)
print('SSL_AUTORESEARCH_CACHE_ROOT:', os.environ['SSL_AUTORESEARCH_CACHE_ROOT'])
print('SSL_AUTORESEARCH_OUTPUT_ROOT:', os.environ['SSL_AUTORESEARCH_OUTPUT_ROOT'])


In [ ]:
# Imports.

import importlib
import sys
import time
from pathlib import Path

import torch

# Colab can keep a stale possm_ssl module cached across git pulls.
for module_name in list(sys.modules):
    if module_name == 'possm_ssl' or module_name.startswith('possm_ssl.'):
        sys.modules.pop(module_name, None)
importlib.invalidate_caches()

from possm_ssl import (
    CacheAccessConfig,
    POSSMFinetuneConfig,
    POSSM_B2T24_B2T25_SBP,
    POSSM_B2T24_SBP,
    POSSMTrainingConfig,
    display_possm_stage1_report,
    display_possm_stage2_report,
    display_possm_stage2_summary,
    load_precomputed_session_feature_stats_into_cache_context,
    prepare_cache_context,
    recover_possm_run_state_from_checkpoint,
    recover_possm_stage2_summary,
    resolve_latest_possm_checkpoint_path,
    resolve_possm_checkpoint_path,
    resume_possm_training,
    run_possm_phoneme_finetuning,
    run_possm_stage1_prediction_diagnostics,
    run_possm_stage2_prediction_diagnostics,
    run_possm_training,
)
from ssl_core.stats import resolve_precomputed_split_stats_path
from masked_ssl.cache import (
    SESSION_STATS_BIN_STRIDE,
    _cache_variant_name,
    _compute_dataset_cache_source_signature,
    resolve_boundary_key,
    resolve_precomputed_session_stats_path,
    stable_text_seed,
)
import possm_ssl.phoneme_finetune as possm_phoneme_finetune

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)
print('GPU:', torch.cuda.get_device_name(DEVICE) if DEVICE.type == 'cuda' else None)
NOTEBOOK_T0 = time.perf_counter()

_POSSM_ORIGINAL_EMIT_PROGRESS = getattr(
    possm_phoneme_finetune,
    '_notebook_original_emit_progress',
    possm_phoneme_finetune._emit_progress,
)
possm_phoneme_finetune._notebook_original_emit_progress = _POSSM_ORIGINAL_EMIT_PROGRESS

def _fmt_progress_metric(value, digits=3):
    if value is None:
        return 'nan'
    try:
        return f'{float(value):.{digits}f}'
    except (TypeError, ValueError):
        return str(value)

def _emit_possm_progress_with_stdout(progress_log_path, **payload):
    _POSSM_ORIGINAL_EMIT_PROGRESS(progress_log_path, **payload)
    event = str(payload.get('event', 'progress'))
    step = payload.get('step')
    elapsed = _fmt_progress_metric(payload.get('elapsed_seconds'), digits=1)
    if event == 'phoneme_resume':
        print(f'[stage2 resume] step={step} from={payload.get("resumed_from_checkpoint")} elapsed_s={elapsed}')
    elif event == 'phoneme_train_report':
        train_loss = _fmt_progress_metric(payload.get('train_ctc_bpphone'))
        sample_s = _fmt_progress_metric(payload.get('sample_seconds'), digits=2)
        model_s = _fmt_progress_metric(payload.get('model_seconds'), digits=2)
        print(
            f'[stage2 train] step={step} train_ctc_bpphone={train_loss} '
            f'sample_s={sample_s} model_s={model_s} elapsed_s={elapsed}'
        )
    elif event == 'phoneme_val_report':
        val_loss = _fmt_progress_metric(payload.get('val_ctc_bpphone'))
        val_per = _fmt_progress_metric(payload.get('val_phoneme_error_rate'))
        collapse = payload.get('collapse_diagnostics') or {}
        blank_rate = _fmt_progress_metric(collapse.get('blank_frame_rate'))
        pred_ratio = _fmt_progress_metric(collapse.get('predicted_to_reference_token_ratio'))
        print(
            f'[stage2 val] step={step} val_ctc_bpphone={val_loss} '
            f'val_PER={val_per} pred/ref={pred_ratio} blank={blank_rate} elapsed_s={elapsed}'
        )

possm_phoneme_finetune._emit_progress = _emit_possm_progress_with_stdout


In [ ]:
# Experiment configuration. Edit this cell only.

SEED = 7
PRECISION = 'amp_fp16'
STAGE1_RECIPE = POSSM_B2T24_SBP
STAGE1_SIGNAL_SPEC = STAGE1_RECIPE.signal_spec
STAGE1_DATASET_PLAN = STAGE1_RECIPE.dataset_plan
FEATURE_MODE = STAGE1_SIGNAL_SPEC.mode
BOUNDARY_KEY_MODE = 'session'
print('requested precision:', PRECISION)
SEGMENT_BINS = 100
USE_NORMALIZATION = True
USE_SMOOTHED_CACHE = True
STAGE1_CACHE_MODE = 'drive_direct'  # use 'copy_to_local' only if warm Drive sampling remains slow

STAGE1_PRIMARY_CACHE_ROOT = DEFAULT_CACHE_ROOT_SMOOTHED if USE_SMOOTHED_CACHE else DEFAULT_CACHE_ROOT_RAW
STAGE1_B2T25_CACHE_ROOT = STAGE1_B2T25_CACHE_ROOT_SMOOTHED if USE_SMOOTHED_CACHE else STAGE1_B2T25_CACHE_ROOT_RAW
if 'brain2text25' in STAGE1_DATASET_PLAN.dataset_names and not STAGE1_B2T25_CACHE_ROOT.is_dir():
    raise FileNotFoundError(
        'Optimized Brain2Text25 Stage-1 cache is missing. Run '
        'notebooks/s15_possm_pooled_cache_preparation.ipynb first: '
        f'{STAGE1_B2T25_CACHE_ROOT}'
    )
STAGE1_DATASET_CACHE_ROOTS = (
    {'brain2text25': STAGE1_B2T25_CACHE_ROOT}
    if 'brain2text25' in STAGE1_DATASET_PLAN.dataset_names
    else {}
)
if STAGE1_CACHE_MODE not in {'drive_direct', 'copy_to_local'}:
    raise ValueError("STAGE1_CACHE_MODE must be one of {'drive_direct', 'copy_to_local'}")

# Exact Stage-1 source plan. Brain2Text24 test data are intentionally absent.
STAGE1_SOURCE_SPLITS_BY_DATASET = STAGE1_DATASET_PLAN.source_splits_by_dataset
STAGE1_DATASETS = STAGE1_DATASET_PLAN.dataset_names
STAGE1_NORMALIZATION_SCOPE = 'session'
GAUSSIAN_SMOOTHING_SIGMA_BINS = 0.0

# Stage 1: train once, then resume this named run if Colab stops.
STAGE1_STATE_MODE = 'train'  # one of {'train', 'resume', 'recover'}
STAGE1_RUN_NAME = 'possm_stage1_sbp_b2t24_only_12k_seed7_amp_fp16_v1'
STAGE1_OUTPUT_SUBDIR = 'sbp_only_b2t24_amp_fp16_v1'
STAGE1_RESUME_TARGET_STEPS = 12000
STAGE1_RECOVERY_RUN_DIR = None
STAGE1_RECOVERY_EXPLICIT_CHECKPOINT_PATH = None

# Stage 2: keep the baseline recipe and 12k-step compute budget.
DATASET = 'brain2text24'
STAGE2_CONFIG_PRESET = 'current_finetune_full'
STAGE2_STATE_MODE = 'finetune_full'
STAGE2_INIT_SOURCE = 'stage1'
STAGE2_RUN_ACTION = 'fresh'  # one of {'fresh', 'resume_latest', 'recover_only', 'skip'}
STAGE2_OUTPUT_SUBDIR = 'sbp_only_b2t24_amp_fp16_v1_stage2'
STAGE2_RECOVERY_RUN_DIR_OVERRIDE = None
STAGE2_RECOVERY_CHECKPOINT_OVERRIDE = None
STAGE2_SESSION_ADAPTER_ENABLED = False
STAGE2_INPUT_SOURCE = 'raw_online_smoothing'
STAGE2_EMISSION_MODE = 'post_decoder_conv'
STAGE2_PRE_DECODER_PATCH_SIZE = 14
STAGE2_PRE_DECODER_PATCH_STRIDE = 4
STAGE2_SCREEN_NUM_STEPS = 12000
STAGE2_SCREEN_VAL_EVERY_STEPS = 100
STAGE2_SCREEN_CHECKPOINT_EVERY_STEPS = 500

STAGE1_RUN_STATE = None
STAGE1_RECOVERED_CHECKPOINT_PATH = None
STAGE2_STAGE1_CHECKPOINT_OVERRIDE = None
RUN_STAGE1_PREDICTION_DIAGNOSTICS = False
RUN_STAGE2_PREDICTION_DIAGNOSTICS = False

if set(STAGE1_SOURCE_SPLITS_BY_DATASET) != set(STAGE1_DATASETS):
    raise ValueError('STAGE1_DATASETS must match the explicit Stage-1 split policy.')
if 'competition_test' in set(STAGE1_SOURCE_SPLITS_BY_DATASET.get('brain2text24', ())):
    raise ValueError('Brain2Text24 competition_test must not enter Stage-1 pretraining.')
if 'test' in set(STAGE1_SOURCE_SPLITS_BY_DATASET.get('brain2text25', ())):
    raise ValueError('Brain2Text25 test must not enter Stage-1 pretraining.')

print({
    'stage1_datasets': STAGE1_DATASETS,
    'stage1_source_splits_by_dataset': STAGE1_SOURCE_SPLITS_BY_DATASET,
    'stage1_state_mode': STAGE1_STATE_MODE,
    'stage2_run_action': STAGE2_RUN_ACTION,
    'stage2_dataset': DATASET,
    'stage2_steps': STAGE2_SCREEN_NUM_STEPS,
})


In [ ]:
# Reuse or recompute canonical stats for the active config.
# Set FORCE_RECOMPUTE_STATS=True after changing cache contents or when intentionally regenerating artifacts.

from pathlib import Path
import json
import subprocess
import sys

from ssl_core.stats import resolve_precomputed_split_stats_path

FORCE_RECOMPUTE_STATS = False


def _run_stats_command(label, cmd):
    print(f'Running {label}:', ' '.join(str(part) for part in cmd))
    result = subprocess.run(cmd, cwd=str(REPO_DIR), text=True, capture_output=True)
    print(f'{label} returncode:', result.returncode)
    if result.stdout:
        print(f'\n{label} STDOUT\n')
        print(result.stdout)
    if result.stderr:
        print(f'\n{label} STDERR\n')
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'{label} recompute failed.')


def _print_metadata(path):
    metadata_path = Path(path).with_suffix('.json')
    print('artifact:', path)
    print('sidecar:', metadata_path)
    if metadata_path.exists():
        metadata = json.loads(metadata_path.read_text())
        preview_keys = [
            'kind',
            'source_cache_variant',
            'dataset',
            'feature_mode',
            'boundary_key_mode',
            'full_dim',
            'feature_dim',
            'signal_spec',
            'dataset_plan',
        ]
        print(json.dumps({key: metadata.get(key) for key in preview_keys if key in metadata}, indent=2))


def _stats_artifact_complete(path):
    path = Path(path)
    metadata_path = path.with_suffix('.json')
    if not path.exists() or not metadata_path.exists():
        return False
    try:
        metadata = json.loads(metadata_path.read_text())
    except (OSError, json.JSONDecodeError):
        return False
    is_session_stats = 'session_feature_stats' in str(path)
    if metadata.get('signal_spec') != STAGE1_SIGNAL_SPEC.to_dict():
        return False
    if is_session_stats:
        expected_cache_root = stage1_cache_root_for_stats
    else:
        expected_cache_root = stage2_cache_root_for_stats
    if is_session_stats:
        expected_cache_roots = {
            dataset: STAGE1_DATASET_CACHE_ROOTS.get(
                dataset, stage1_cache_root_for_stats
            )
            for dataset in STAGE1_DATASETS
        }
        expected_signature = _compute_dataset_cache_source_signature(expected_cache_roots)
    else:
        expected_signature = _compute_dataset_cache_source_signature({DATASET: expected_cache_root})
    if metadata.get('source_cache_signature') != expected_signature:
        return False
    if is_session_stats:
        return metadata.get('dataset_plan') == STAGE1_DATASET_PLAN.to_dict()
    return True


def _ensure_stats_artifact(label, path, cmd):
    path = Path(path)
    metadata_path = path.with_suffix('.json')
    if _stats_artifact_complete(path) and not FORCE_RECOMPUTE_STATS:
        print(f'Reusing existing {label}.')
        _print_metadata(path)
        return

    recompute_cmd = list(cmd)
    if FORCE_RECOMPUTE_STATS or path.exists() or metadata_path.exists():
        recompute_cmd.append('--overwrite')
    _run_stats_command(label, recompute_cmd)
    _print_metadata(path)


if STAGE1_NORMALIZATION_SCOPE != 'session':
    raise ValueError("STAGE1_NORMALIZATION_SCOPE must be 'session' for the canonical stats flow.")
if STAGE2_INPUT_SOURCE not in {'smoothed_cache', 'raw_online_smoothing'}:
    raise ValueError("STAGE2_INPUT_SOURCE must be one of {'smoothed_cache', 'raw_online_smoothing'}")

stage1_session_stats_path = None
stage1_cache_root_for_stats = DEFAULT_CACHE_ROOT_SMOOTHED if USE_SMOOTHED_CACHE else DEFAULT_CACHE_ROOT_RAW
if USE_NORMALIZATION and STAGE1_NORMALIZATION_SCOPE == 'session':
    stage1_session_stats_path = resolve_precomputed_session_stats_path(
        cache_root=stage1_cache_root_for_stats,
        signal_spec=STAGE1_SIGNAL_SPEC,
        dataset_plan=STAGE1_DATASET_PLAN,
        boundary_key_mode=BOUNDARY_KEY_MODE,
        dataset_cache_roots=STAGE1_DATASET_CACHE_ROOTS or None,
    )
    cmd = [
        sys.executable,
        str(REPO_DIR / 'analysis/active/ssl_experiments/ssl_core/scripts/recompute_feature_stats.py'),
        '--scope', 'session',
        '--cache-root', str(stage1_cache_root_for_stats),
        '--output-path', str(stage1_session_stats_path),
        '--feature-mode', FEATURE_MODE,
        '--boundary-key-mode', BOUNDARY_KEY_MODE,
        '--tx-dim', str(STAGE1_SIGNAL_SPEC.tx_dim),
        '--sbp-dim', str(STAGE1_SIGNAL_SPEC.sbp_dim),
        '--column-start', str(STAGE1_SIGNAL_SPEC.column_start),
        '--missing-channel-policy', STAGE1_SIGNAL_SPEC.missing_channel_policy,
    ]
    for dataset, dataset_cache_root in STAGE1_DATASET_CACHE_ROOTS.items():
        cmd.extend([
            '--dataset-cache-root',
            f'{dataset}={dataset_cache_root}',
        ])
    for dataset in STAGE1_DATASETS:
        cmd.extend(['--dataset', dataset])
    for dataset, source_splits in STAGE1_SOURCE_SPLITS_BY_DATASET.items():
        for source_split in source_splits:
            cmd.extend(['--dataset-source-split', f'{dataset}={source_split}'])
    _ensure_stats_artifact('stage1_session_stats', stage1_session_stats_path, cmd)
stage2_cache_root_for_stats = (
    DEFAULT_CACHE_ROOT_SMOOTHED
    if STAGE2_INPUT_SOURCE == 'smoothed_cache'
    else DEFAULT_CACHE_ROOT_RAW
)
stage2_split_stats_path = resolve_precomputed_split_stats_path(
    cache_root=stage2_cache_root_for_stats,
    dataset=DATASET,
    train_split_name='competition_train',
    signal_spec=STAGE1_SIGNAL_SPEC,
    preferred_path=None,
)
cmd = [
    sys.executable,
    str(REPO_DIR / 'analysis/active/ssl_experiments/ssl_core/scripts/recompute_feature_stats.py'),
    '--scope', 'global',
    '--cache-root', str(stage2_cache_root_for_stats),
    '--dataset', DATASET,
    '--feature-mode', FEATURE_MODE,
    '--tx-dim', str(STAGE1_SIGNAL_SPEC.tx_dim),
    '--sbp-dim', str(STAGE1_SIGNAL_SPEC.sbp_dim),
    '--column-start', str(STAGE1_SIGNAL_SPEC.column_start),
    '--missing-channel-policy', STAGE1_SIGNAL_SPEC.missing_channel_policy,
    '--boundary-key-mode', BOUNDARY_KEY_MODE,
    '--split-policy', 'competition_train_test',
    '--output-path', str(stage2_split_stats_path),
]
_ensure_stats_artifact('stage2_split_stats', stage2_split_stats_path, cmd)


In [ ]:
# Resolve cache root and build cache context.

CACHE_ROOT = STAGE1_PRIMARY_CACHE_ROOT
if STAGE1_NORMALIZATION_SCOPE != 'session':
    raise ValueError("STAGE1_NORMALIZATION_SCOPE must be 'session' for the canonical stats flow.")

CACHE_ACCESS_CONFIG = CacheAccessConfig(
    mode=STAGE1_CACHE_MODE,
    local_cache_base='/content/utah_ssl_cache',
    force_recopy_local_cache=False,
    dataset_plan=STAGE1_DATASET_PLAN,
    signal_spec=STAGE1_SIGNAL_SPEC,
    seed=SEED,
    segment_bins=SEGMENT_BINS,
    use_normalization=bool(USE_NORMALIZATION),
    examples_per_shard=8,
    boundary_key_mode=BOUNDARY_KEY_MODE,
    gaussian_smoothing_sigma_bins=GAUSSIAN_SMOOTHING_SIGMA_BINS,
    precomputed_session_stats_path=stage1_session_stats_path,
    dataset_cache_roots=STAGE1_DATASET_CACHE_ROOTS or None,
)

CACHE_BUILD_T0 = time.perf_counter()
CACHE_CONTEXT = prepare_cache_context(cache_candidates=[CACHE_ROOT], config=CACHE_ACCESS_CONFIG)

print('CACHE_ROOT:', CACHE_ROOT)
print('STAGE1_DATASET_CACHE_ROOTS:', STAGE1_DATASET_CACHE_ROOTS)
print('CACHE_CONTEXT.cache_root:', CACHE_CONTEXT.cache_root)
print('CACHE_CONTEXT.feature_mode:', CACHE_CONTEXT.feature_mode)
print('CACHE_CONTEXT.boundary_key_mode:', CACHE_CONTEXT.boundary_key_mode)
print('CACHE_CONTEXT.use_normalization:', CACHE_CONTEXT.use_normalization)
print('CACHE_CONTEXT.pretrain_datasets:', CACHE_CONTEXT.pretrain_datasets)
print('CACHE_CONTEXT.dataset_plan:', CACHE_CONTEXT.config.dataset_plan.to_dict())
print('CACHE_CONTEXT.shard_store:', CACHE_CONTEXT.shard_store.summary())
print('STAGE1_NORMALIZATION_SCOPE:', STAGE1_NORMALIZATION_SCOPE)
print('CACHE_CONTEXT build elapsed_s:', round(time.perf_counter() - CACHE_BUILD_T0, 1))


In [ ]:
# Stage-1 POSSM config.

STAGE1_DATA_MODE = 'normalized' if USE_NORMALIZATION else 'raw'
STAGE1_CONFIG = POSSMTrainingConfig(
    seed=SEED,
    precision=PRECISION,
    data_mode=STAGE1_DATA_MODE,
    signal_spec=STAGE1_SIGNAL_SPEC,
    boundary_key_mode=BOUNDARY_KEY_MODE,
    segment_bins=SEGMENT_BINS,
    model_dim=64,
    latent_count=4,
    value_encoder_type='linear',
    value_mlp_hidden_size=None,
    ffn_hidden_size=512,
    dropout=0.15,
    use_token_norm=True,
    batch_size=32,
    num_steps=STAGE1_RESUME_TARGET_STEPS,
    learning_rate=3e-4,
    weight_decay=1e-3,
    val_every=50,
    val_batches=2,
    checkpoint_every_steps=1000,
    checkpoint_keep_last=2,
    dataset_weight_alpha=0.25,
    examples_per_shard=8,
    log_every=20,
    temporal_backbone_type='gru',
    temporal_gru_hidden_size=None,
    temporal_gru_num_layers=1,
    temporal_gru_dropout=0.0,
    temporal_gru_bidirectional=False,
    temporal_backbone_kwargs={},
    stage1_objective_type='plain_mse',
    masking_type='none',
    mask_prob=0.0,
    mask_span_bins=8,
    mask_replace_mode='zero',
    reconstruction_head_type='linear',
    reconstruction_mlp_hidden_size=None,
)
print(STAGE1_CONFIG)


In [ ]:
# Stage-1 run/recover/resume.

STAGE1_T0 = time.perf_counter()
STAGE1_RECOVERED_CHECKPOINT_PATH = None
STAGE1_OUTPUT_ROOT = OUTPUT_ROOT / STAGE1_OUTPUT_SUBDIR
STAGE1_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if STAGE1_STATE_MODE == 'train':
    STAGE1_RUN_STATE = run_possm_training(
        cache_context=CACHE_CONTEXT,
        config=STAGE1_CONFIG,
        output_root=STAGE1_OUTPUT_ROOT,
        device=DEVICE,
        run_name=STAGE1_RUN_NAME,
    )
elif STAGE1_STATE_MODE == 'recover':
    stage1_checkpoint = resolve_possm_checkpoint_path(
        output_root=STAGE1_OUTPUT_ROOT,
        run_dir=STAGE1_RECOVERY_RUN_DIR,
        explicit_checkpoint_path=STAGE1_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
    )
    STAGE1_RECOVERED_CHECKPOINT_PATH = stage1_checkpoint
    STAGE1_RUN_STATE = recover_possm_run_state_from_checkpoint(
        cache_context=CACHE_CONTEXT,
        checkpoint_path=stage1_checkpoint,
        device=DEVICE,
    )
elif STAGE1_STATE_MODE in {'resume', 'resume_latest'}:
    resolver = resolve_latest_possm_checkpoint_path if STAGE1_STATE_MODE == 'resume_latest' else resolve_possm_checkpoint_path
    stage1_checkpoint = resolver(
        output_root=STAGE1_OUTPUT_ROOT,
        run_dir=STAGE1_RECOVERY_RUN_DIR,
        explicit_checkpoint_path=STAGE1_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
    )
    STAGE1_RECOVERED_CHECKPOINT_PATH = stage1_checkpoint
    STAGE1_RUN_STATE = recover_possm_run_state_from_checkpoint(
        cache_context=CACHE_CONTEXT,
        checkpoint_path=stage1_checkpoint,
        device=DEVICE,
    )
    current_step = int(STAGE1_RUN_STATE['checkpoint_step'])
    additional_steps = max(0, int(STAGE1_RESUME_TARGET_STEPS) - current_step)
    print('Stage-1 resume:', current_step, '->', STAGE1_RESUME_TARGET_STEPS, 'steps')
    if additional_steps:
        STAGE1_RUN_STATE = resume_possm_training(
            run_state=STAGE1_RUN_STATE, additional_steps=additional_steps,
            cache_context=CACHE_CONTEXT, device=DEVICE,
        )
else:
    raise ValueError('STAGE1_STATE_MODE must be one of {train, recover, resume, resume_latest}')

print('stage1 run_dir:', STAGE1_RUN_STATE['run_dir'])
print('stage1 precision:', STAGE1_RUN_STATE['precision_runtime'].metadata())
print('stage1 optimizer fused:', STAGE1_RUN_STATE.get('optimizer_fused'))
print('stage1 final checkpoint:', STAGE1_RUN_STATE['checkpoint_path'])
print('stage1 best checkpoint:', STAGE1_RUN_STATE['best_checkpoint_path'])
print('stage1 elapsed_s:', round(time.perf_counter() - STAGE1_T0, 1))


In [ ]:
# Optional Stage-1 report / diagnostics.

STAGE1_REPORT = display_possm_stage1_report(STAGE1_RUN_STATE)
if RUN_STAGE1_PREDICTION_DIAGNOSTICS:
    STAGE1_PREDICTION_DIAGNOSTICS = run_possm_stage1_prediction_diagnostics(
        STAGE1_RUN_STATE,
        device=DEVICE,
    )


In [ ]:
# Stage-2 config + checkpoint handoff.

if STAGE2_RUN_ACTION == 'recover_only':
    STAGE2_STAGE1_CHECKPOINT = None
    STAGE2_CHECKPOINT_SOURCE = 'unused for recover_only'
elif STAGE2_STAGE1_CHECKPOINT_OVERRIDE is not None:
    STAGE2_STAGE1_CHECKPOINT = Path(STAGE2_STAGE1_CHECKPOINT_OVERRIDE)
    STAGE2_CHECKPOINT_SOURCE = 'explicit override'
elif isinstance(globals().get('STAGE1_RUN_STATE'), dict) and 'best_checkpoint_path' in STAGE1_RUN_STATE:
    STAGE2_STAGE1_CHECKPOINT = Path(STAGE1_RUN_STATE['best_checkpoint_path'])
    STAGE2_CHECKPOINT_SOURCE = 'in-memory STAGE1_RUN_STATE best checkpoint'
elif STAGE1_RECOVERED_CHECKPOINT_PATH is not None:
    STAGE2_STAGE1_CHECKPOINT = Path(STAGE1_RECOVERED_CHECKPOINT_PATH)
    STAGE2_CHECKPOINT_SOURCE = 'explicit Stage-1 recovery checkpoint'
else:
    STAGE2_STAGE1_CHECKPOINT = resolve_possm_checkpoint_path(
        run_dir=STAGE1_OUTPUT_ROOT / STAGE1_RUN_NAME,
    )
    STAGE2_CHECKPOINT_SOURCE = 'named SBP-only Stage-1 run directory'

if STAGE2_STAGE1_CHECKPOINT is not None and not STAGE2_STAGE1_CHECKPOINT.exists():
    raise FileNotFoundError(f'Stage-1 checkpoint not found: {STAGE2_STAGE1_CHECKPOINT}')
if STAGE2_INIT_SOURCE not in {'stage1', 'random'}:
    raise ValueError("STAGE2_INIT_SOURCE must be one of {'stage1', 'random'}")

if STAGE2_INPUT_SOURCE == 'smoothed_cache':
    STAGE2_CACHE_ROOT = DEFAULT_CACHE_ROOT_SMOOTHED
    STAGE2_INPUT_SMOOTHING_SIGMA_BINS = 0.0
elif STAGE2_INPUT_SOURCE == 'raw_online_smoothing':
    STAGE2_CACHE_ROOT = DEFAULT_CACHE_ROOT_RAW
    STAGE2_INPUT_SMOOTHING_SIGMA_BINS = 2.0
else:
    raise ValueError("STAGE2_INPUT_SOURCE must be one of {'smoothed_cache', 'raw_online_smoothing'}")

if not STAGE2_CACHE_ROOT.exists():
    raise FileNotFoundError(f'Stage-2 cache root not found: {STAGE2_CACHE_ROOT}')

STAGE2_SPLIT_STATS_PATH = resolve_precomputed_split_stats_path(
    cache_root=STAGE2_CACHE_ROOT,
    dataset=DATASET,
    train_split_name='competition_train',
    signal_spec=STAGE1_SIGNAL_SPEC,
    preferred_path=None,
)
if not STAGE2_SPLIT_STATS_PATH.exists():
    raise FileNotFoundError(
        'Stage-2 split stats are missing. Run the canonical stats helper cell first: '
        f'{STAGE2_SPLIT_STATS_PATH}'
    )

if STAGE2_CONFIG_PRESET == 'historical_probe_frozen':
    stage2_config_payload = dict(
        mode='probe_frozen',
        weight_decay=1e-3,
        session_adapter_enabled=False,
    )
elif STAGE2_CONFIG_PRESET == 'current_finetune_full':
    stage2_config_payload = dict(
        mode='probe_frozen' if STAGE2_STATE_MODE == 'probe_frozen' else 'finetune_full',
        weight_decay=1e-3,
        session_adapter_enabled=STAGE2_SESSION_ADAPTER_ENABLED,
    )
else:
    raise ValueError("STAGE2_CONFIG_PRESET must be one of {'historical_probe_frozen', 'current_finetune_full'}")

STAGE2_CONFIG = POSSMFinetuneConfig(
    seed=SEED,
    precision=PRECISION,
    init_source=STAGE2_INIT_SOURCE,
    dataset=DATASET,
    signal_spec=STAGE1_SIGNAL_SPEC,
    data_mode=STAGE1_DATA_MODE,
    boundary_key_mode=BOUNDARY_KEY_MODE,
    batch_size=32,
    num_steps=STAGE2_SCREEN_NUM_STEPS,
    learning_rate=2e-4,
    encoder_learning_rate=3e-5,
    max_grad_norm=1.0,
    val_every_steps=STAGE2_SCREEN_VAL_EVERY_STEPS,
    checkpoint_every_steps=STAGE2_SCREEN_CHECKPOINT_EVERY_STEPS,
    checkpoint_keep_last=2,
    progress_every_steps=20,
    benchmark_warmup_steps=20,
    benchmark_measure_steps=100,
    input_smoothing_sigma_bins=STAGE2_INPUT_SMOOTHING_SIGMA_BINS,
    input_smoothing_kernel_size=100,
    input_smoothing_threshold=0.01,
    white_noise_sd=0.1,
    constant_offset_sd=0.05,
    precomputed_split_stats_path=STAGE2_SPLIT_STATS_PATH,
    decoder_backbone_type='gru',
    gru_hidden_size=768, gru_num_layers=5, gru_dropout=0.2,
    emission_mode=STAGE2_EMISSION_MODE,
    pre_decoder_patch_size=STAGE2_PRE_DECODER_PATCH_SIZE,
    pre_decoder_patch_stride=STAGE2_PRE_DECODER_PATCH_STRIDE,
    **stage2_config_payload,
)

print('STAGE2_RUN_ACTION:', STAGE2_RUN_ACTION)
print('STAGE2_CONFIG_PRESET:', STAGE2_CONFIG_PRESET)
print('STAGE2_INIT_SOURCE:', STAGE2_INIT_SOURCE)
print('STAGE2_INPUT_SOURCE:', STAGE2_INPUT_SOURCE)
print('STAGE2_EMISSION_MODE:', STAGE2_EMISSION_MODE)
print('STAGE2_PRE_DECODER_PATCH:', (STAGE2_PRE_DECODER_PATCH_SIZE, STAGE2_PRE_DECODER_PATCH_STRIDE))
print('STAGE2_CHECKPOINT_SOURCE:', STAGE2_CHECKPOINT_SOURCE)
print('STAGE2_STAGE1_CHECKPOINT:', STAGE2_STAGE1_CHECKPOINT)
print('STAGE2_CACHE_ROOT:', STAGE2_CACHE_ROOT)
print('STAGE2_SPLIT_STATS_PATH:', STAGE2_SPLIT_STATS_PATH)
print('STAGE2_INPUT_SMOOTHING_SIGMA_BINS:', STAGE2_INPUT_SMOOTHING_SIGMA_BINS)
print(STAGE2_CONFIG)


In [ ]:
# Colab/PyTorch 2.6 compatibility for our trusted local checkpoints.
_ORIGINAL_TORCH_LOAD = getattr(torch, '_possm_original_torch_load', torch.serialization.load)
torch._possm_original_torch_load = _ORIGINAL_TORCH_LOAD

def _torch_load_weights_only_false(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _ORIGINAL_TORCH_LOAD(*args, **kwargs)

torch.load = _torch_load_weights_only_false

In [ ]:
print("decoder:", STAGE2_CONFIG.decoder_backbone_type)
print("init_source:", STAGE2_CONFIG.init_source)
print("emission_mode:", STAGE2_CONFIG.emission_mode)
print("pre_decoder_patch:", (STAGE2_CONFIG.pre_decoder_patch_size, STAGE2_CONFIG.pre_decoder_patch_stride))


In [ ]:
# Stage-2 run, recover, or resume.

STAGE2_T0 = time.perf_counter()
STAGE2_OUTPUT_ROOT = OUTPUT_ROOT / STAGE2_OUTPUT_SUBDIR
STAGE2_SUMMARY = None
RECOVERED_STAGE2_SUMMARY = None

if STAGE2_RUN_ACTION == 'skip':
    print('Skipping stage-2.')
elif STAGE2_RUN_ACTION == 'fresh':
    STAGE2_SUMMARY = run_possm_phoneme_finetuning(
        checkpoint_path=STAGE2_STAGE1_CHECKPOINT,
        cache_root=STAGE2_CACHE_ROOT,
        output_root=STAGE2_OUTPUT_ROOT,
        config=STAGE2_CONFIG,
        device=DEVICE,
    )
elif STAGE2_RUN_ACTION in {'recover_only', 'resume_latest'}:
    RECOVERED_STAGE2_SUMMARY = recover_possm_stage2_summary(
        STAGE2_OUTPUT_ROOT,
        run_dir=STAGE2_RECOVERY_RUN_DIR_OVERRIDE,
        checkpoint_path=STAGE2_RECOVERY_CHECKPOINT_OVERRIDE,
    )
    print('recovered stage2 run_dir:', RECOVERED_STAGE2_SUMMARY['run_dir'])
    print('resume checkpoint:', RECOVERED_STAGE2_SUMMARY['resume_checkpoint_path'])
    if STAGE2_RUN_ACTION == 'recover_only':
        STAGE2_SUMMARY = RECOVERED_STAGE2_SUMMARY
    else:
        recovered_config = POSSMFinetuneConfig(**RECOVERED_STAGE2_SUMMARY['config'])
        STAGE2_SUMMARY = run_possm_phoneme_finetuning(
            checkpoint_path=RECOVERED_STAGE2_SUMMARY['stage1_checkpoint_path'],
            cache_root=RECOVERED_STAGE2_SUMMARY['cache_root'],
            output_root=STAGE2_OUTPUT_ROOT,
            config=recovered_config,
            device=DEVICE,
            run_name=RECOVERED_STAGE2_SUMMARY['run_name'],
            resume_from_latest=True,
        )
else:
    raise ValueError("STAGE2_RUN_ACTION must be one of {'fresh', 'resume_latest', 'recover_only', 'skip'}")

if STAGE2_SUMMARY is not None:
    print('stage2 run_dir:', STAGE2_SUMMARY['run_dir'])
    print('stage2 requested/resolved precision:', STAGE2_SUMMARY.get('requested_precision'), STAGE2_SUMMARY.get('resolved_precision'))
    print('stage2 AMP/scaler:', STAGE2_SUMMARY.get('amp_enabled'), STAGE2_SUMMARY.get('grad_scaler_enabled'))
    print('stage2 optimizer fused:', STAGE2_SUMMARY.get('optimizer_fused'))
    print('stage2 resumed_from_checkpoint:', STAGE2_SUMMARY.get('resumed_from_checkpoint'))
    print('stage2 best checkpoint:', STAGE2_SUMMARY.get('checkpoint_best_path'))
    print('stage2 final checkpoint:', STAGE2_SUMMARY.get('checkpoint_final_path'))
print('stage2 elapsed_s:', round(time.perf_counter() - STAGE2_T0, 1))


In [ ]:
# Optional Stage-2 report / diagnostics.

STAGE2_REPORT = display_possm_stage2_report(STAGE2_SUMMARY)
STAGE2_SUMMARY_FRAMES = display_possm_stage2_summary(STAGE2_SUMMARY)
if RUN_STAGE2_PREDICTION_DIAGNOSTICS and STAGE2_SUMMARY is not None:
    STAGE2_PREDICTION_DIAGNOSTICS = run_possm_stage2_prediction_diagnostics(
        STAGE2_SUMMARY,
        device=DEVICE,
    )
print('notebook elapsed_s:', round(time.perf_counter() - NOTEBOOK_T0, 1))


In [ ]:
from google.colab import runtime
runtime.unassign()